In [2]:
# ============================================================
# URBAN MOBILITY INTELLIGENCE SYSTEM
# Notebook 02 — PostgreSQL Database Loading
# ============================================================

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import psycopg2
import os
import warnings
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────
RAW_DATA_PATH = r"D:\Projects\End-to-end projects\14. Urban Mobility Intelligence\data\raw"

# ── PostgreSQL Connection Config ─────────────────────────────
DB_CONFIG = {
    "host":     "localhost",
    "port":     5432,
    "user":     "postgres",
    "password": "NewStrongPassword@123",   # your password (no need to encode here)
    "database": "postgres"        # connect to default db first
}

DB_NAME = "urban_mobility_db"

# ============================================================
# BLOCK 1 — Test psycopg2 Connection
# ============================================================
try:
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    cursor = conn.cursor()

    cursor.execute("SELECT version();")
    version = cursor.fetchone()[0]

    print("=" * 60)
    print("  PostgreSQL CONNECTION SUCCESSFUL")
    print("=" * 60)
    print(f"  {version[:60]}")
    print(f"  Host     : {DB_CONFIG['host']}")
    print(f"  Port     : {DB_CONFIG['port']}")
    print(f"  User     : {DB_CONFIG['user']}")
    print("=" * 60)

    print("\n✓ Block 1 complete — PostgreSQL connected.")

except Exception as e:
    print(f"✗ Connection failed: {e}")
    print("\nCheck your password or PostgreSQL service.")

# ============================================================
# BLOCK 2 — Create Database (if not exists)
# ============================================================
try:
    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'")
    exists = cursor.fetchone()

    if not exists:
        cursor.execute(f"CREATE DATABASE {DB_NAME}")
        print(f"✓ Database '{DB_NAME}' created successfully.")
    else:
        print(f"✓ Database '{DB_NAME}' already exists.")

except Exception as e:
    print(f"✗ Error creating database: {e}")

finally:
    cursor.close()
    conn.close()

# ============================================================
# BLOCK 3 — SQLAlchemy Engine (SAFE WAY)
# ============================================================
try:
    db_url = URL.create(
        drivername="postgresql+psycopg2",
        username=DB_CONFIG["user"],
        password=DB_CONFIG["password"],   # no encoding needed
        host=DB_CONFIG["host"],
        port=DB_CONFIG["port"],
        database=DB_NAME
    )

    engine = create_engine(db_url)

    # Test engine
    with engine.connect() as connection:
        result = connection.execute(text("SELECT current_database();"))
        db_connected = result.fetchone()[0]

    print("=" * 60)
    print("  SQLAlchemy ENGINE SUCCESSFUL")
    print("=" * 60)
    print(f"  Connected DB : {db_connected}")
    print("=" * 60)

    print("\n✓ Block 3 complete — SQLAlchemy engine ready.")

except Exception as e:
    print(f"✗ SQLAlchemy connection failed: {e}")

  PostgreSQL CONNECTION SUCCESSFUL
  PostgreSQL 17.6 on x86_64-windows, compiled by msvc-19.44.35
  Host     : localhost
  Port     : 5432
  User     : postgres

✓ Block 1 complete — PostgreSQL connected.
✓ Database 'urban_mobility_db' already exists.
  SQLAlchemy ENGINE SUCCESSFUL
  Connected DB : urban_mobility_db

✓ Block 3 complete — SQLAlchemy engine ready.


In [3]:
# ============================================================
# BLOCK 2 — Create Database & All Table Schemas
# ============================================================

# ── Step 1: Create Database ──────────────────────────────────
try:
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    cursor = conn.cursor()

    # Drop if exists and recreate fresh
    cursor.execute(
        f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}'"
    )
    exists = cursor.fetchone()

    if exists:
        # Terminate active connections first
        cursor.execute(f"""
            SELECT pg_terminate_backend(pid)
            FROM pg_stat_activity
            WHERE datname = '{DB_NAME}'
            AND pid <> pg_backend_pid();
        """)
        cursor.execute(f"DROP DATABASE {DB_NAME};")
        print(f"  Dropped existing database: {DB_NAME}")

    cursor.execute(f"CREATE DATABASE {DB_NAME};")
    print(f"  Created database: {DB_NAME}")
    cursor.close()
    conn.close()

except Exception as e:
    print(f"✗ Database creation failed: {e}")

# ── Step 2: Connect to new database ─────────────────────────
DB_CONFIG_NEW = DB_CONFIG.copy()
DB_CONFIG_NEW["database"] = DB_NAME

from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=DB_CONFIG["port"],
    database=DB_NAME
)

engine = create_engine(db_url)

# ── Step 3: Create All Table Schemas ────────────────────────
schema_sql = """

-- ── 1. ZONES TABLE ─────────────────────────────────────────
CREATE TABLE IF NOT EXISTS zones (
    zone_id         VARCHAR(10)  PRIMARY KEY,
    zone_name       VARCHAR(100) NOT NULL,
    latitude        DECIMAL(9,6) NOT NULL,
    longitude       DECIMAL(9,6) NOT NULL,
    h3_index        VARCHAR(20)  NOT NULL,
    h3_resolution   INTEGER      NOT NULL,
    zone_type       VARCHAR(30)  NOT NULL,
    demand_weight   DECIMAL(4,2) NOT NULL,
    peak_hours      VARCHAR(50),
    city            VARCHAR(50)  NOT NULL,
    is_active       BOOLEAN      DEFAULT TRUE,
    created_at      TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);

-- ── 2. DRIVERS TABLE ───────────────────────────────────────
CREATE TABLE IF NOT EXISTS drivers (
    driver_id            VARCHAR(10)  PRIMARY KEY,
    vehicle_type         VARCHAR(20)  NOT NULL,
    is_ev                BOOLEAN      NOT NULL,
    status               VARCHAR(20)  NOT NULL,
    rating               DECIMAL(3,1),
    experience_months    INTEGER,
    home_zone_id         VARCHAR(10)  REFERENCES zones(zone_id),
    current_zone_id      VARCHAR(10)  REFERENCES zones(zone_id),
    daily_trip_target    INTEGER,
    battery_capacity_kwh DECIMAL(4,1),
    avg_daily_earnings   INTEGER,
    preferred_shift      VARCHAR(20),
    join_date            DATE,
    city                 VARCHAR(50),
    created_at           TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);

-- ── 3. WEATHER TABLE ───────────────────────────────────────
CREATE TABLE IF NOT EXISTS weather (
    weather_id         SERIAL       PRIMARY KEY,
    timestamp          TIMESTAMP    NOT NULL,
    date               DATE         NOT NULL,
    hour               INTEGER      NOT NULL,
    month              INTEGER      NOT NULL,
    temperature_c      DECIMAL(5,1),
    humidity_pct       INTEGER,
    wind_speed_kmh     DECIMAL(5,1),
    rainfall_mm        DECIMAL(6,1),
    is_raining         BOOLEAN,
    rain_intensity     VARCHAR(20),
    visibility_km      DECIMAL(5,1),
    condition          VARCHAR(30),
    demand_multiplier  DECIMAL(4,2),
    city               VARCHAR(50),
    created_at         TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);

-- ── 4. EVENTS TABLE ────────────────────────────────────────
CREATE TABLE IF NOT EXISTS events (
    event_id               VARCHAR(10)  PRIMARY KEY,
    event_name             VARCHAR(150) NOT NULL,
    event_date             DATE         NOT NULL,
    day_of_week            VARCHAR(15),
    is_weekend             BOOLEAN,
    month                  INTEGER,
    venue_zone_id          VARCHAR(10)  REFERENCES zones(zone_id),
    venue_zone_name        VARCHAR(100),
    venue_lat              DECIMAL(9,6),
    venue_lng              DECIMAL(9,6),
    venue_h3_index         VARCHAR(20),
    event_type             VARCHAR(30),
    expected_attendance    INTEGER,
    demand_multiplier      DECIMAL(4,2),
    affected_radius_km     DECIMAL(5,1),
    duration_hours         INTEGER,
    start_hour             INTEGER,
    city                   VARCHAR(50),
    created_at             TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);

-- ── 5. CHARGING STATIONS TABLE ─────────────────────────────
CREATE TABLE IF NOT EXISTS charging_stations (
    station_id           VARCHAR(10)  PRIMARY KEY,
    network              VARCHAR(50)  NOT NULL,
    zone_id              VARCHAR(10)  REFERENCES zones(zone_id),
    zone_name            VARCHAR(100),
    zone_type            VARCHAR(30),
    latitude             DECIMAL(9,6),
    longitude            DECIMAL(9,6),
    h3_index             VARCHAR(20),
    charger_type         VARCHAR(20),
    total_ports          INTEGER,
    power_kw             DECIMAL(6,1),
    cost_per_kwh         DECIMAL(5,1),
    operating_hours      VARCHAR(20),
    is_24hr              BOOLEAN,
    avg_utilization      DECIMAL(4,2),
    monthly_revenue_inr  INTEGER,
    install_year         INTEGER,
    is_operational       BOOLEAN,
    city                 VARCHAR(50),
    created_at           TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);

-- ── 6. RIDES TABLE ─────────────────────────────────────────
CREATE TABLE IF NOT EXISTS rides (
    ride_id               VARCHAR(15)  PRIMARY KEY,
    ride_datetime         TIMESTAMP    NOT NULL,
    date                  DATE         NOT NULL,
    hour                  INTEGER      NOT NULL,
    day_of_week           VARCHAR(15),
    is_weekend            BOOLEAN,
    month                 INTEGER,
    pickup_zone_id        VARCHAR(10)  REFERENCES zones(zone_id),
    pickup_zone_name      VARCHAR(100),
    pickup_zone_type      VARCHAR(30),
    pickup_h3_index       VARCHAR(20),
    dropoff_zone_id       VARCHAR(10)  REFERENCES zones(zone_id),
    dropoff_zone_name     VARCHAR(100),
    dropoff_zone_type     VARCHAR(30),
    dropoff_h3_index      VARCHAR(20),
    driver_id             VARCHAR(10)  REFERENCES drivers(driver_id),
    vehicle_type          VARCHAR(20),
    distance_km           DECIMAL(6,1),
    duration_min          DECIMAL(6,1),
    wait_time_min         DECIMAL(5,1),
    fare_amount           DECIMAL(10,2),
    surge_multiplier      DECIMAL(4,2),
    status                VARCHAR(30),
    payment_mode          VARCHAR(20),
    ride_rating           DECIMAL(3,1),
    cancellation_reason   VARCHAR(50),
    weather_condition     VARCHAR(30),
    rainfall_mm           DECIMAL(6,1),
    is_raining            BOOLEAN,
    temperature_c         DECIMAL(5,1),
    active_event_id       VARCHAR(10),
    active_event_type     VARCHAR(30),
    city                  VARCHAR(50),
    created_at            TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);

-- ── 7. INDEXES FOR QUERY PERFORMANCE ───────────────────────
CREATE INDEX IF NOT EXISTS idx_rides_date
    ON rides(date);
CREATE INDEX IF NOT EXISTS idx_rides_hour
    ON rides(hour);
CREATE INDEX IF NOT EXISTS idx_rides_pickup_zone
    ON rides(pickup_zone_id);
CREATE INDEX IF NOT EXISTS idx_rides_dropoff_zone
    ON rides(dropoff_zone_id);
CREATE INDEX IF NOT EXISTS idx_rides_driver
    ON rides(driver_id);
CREATE INDEX IF NOT EXISTS idx_rides_status
    ON rides(status);
CREATE INDEX IF NOT EXISTS idx_rides_datetime
    ON rides(ride_datetime);
CREATE INDEX IF NOT EXISTS idx_rides_h3_pickup
    ON rides(pickup_h3_index);
CREATE INDEX IF NOT EXISTS idx_weather_date_hour
    ON weather(date, hour);
CREATE INDEX IF NOT EXISTS idx_events_date
    ON events(event_date);
CREATE INDEX IF NOT EXISTS idx_drivers_status
    ON drivers(status);
CREATE INDEX IF NOT EXISTS idx_drivers_zone
    ON drivers(current_zone_id);
"""

try:
    with engine.connect() as conn_eng:
        conn_eng.execute(text(schema_sql))
        conn_eng.commit()

    print("=" * 60)
    print("  DATABASE & SCHEMA CREATION COMPLETE")
    print("=" * 60)
    print(f"  Database : {DB_NAME}")
    print(f"  Tables   : zones, drivers, weather,")
    print(f"             events, charging_stations, rides")
    print(f"  Indexes  : 12 performance indexes created")
    print("=" * 60)
    print("\n✓ Block 2 complete — schema ready.")

except Exception as e:
    print(f"✗ Schema creation failed: {e}")

  Dropped existing database: urban_mobility_db
  Created database: urban_mobility_db
  DATABASE & SCHEMA CREATION COMPLETE
  Database : urban_mobility_db
  Tables   : zones, drivers, weather,
             events, charging_stations, rides
  Indexes  : 12 performance indexes created

✓ Block 2 complete — schema ready.


In [4]:
# ============================================================
# BLOCK 3 - Load All Datasets Into PostgreSQL
# ============================================================

import time

# ── Reload CSVs fresh ────────────────────────────────────────
print("Loading CSVs from disk...")

zones_df    = pd.read_csv(os.path.join(RAW_DATA_PATH, "zones.csv"))
drivers_df  = pd.read_csv(os.path.join(RAW_DATA_PATH, "drivers.csv"))
weather_df  = pd.read_csv(os.path.join(RAW_DATA_PATH, "weather.csv"))
events_df   = pd.read_csv(os.path.join(RAW_DATA_PATH, "events.csv"))
stations_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "charging_stations.csv"))
rides_df    = pd.read_csv(os.path.join(RAW_DATA_PATH, "rides.csv"))

print(f"  ✓ All 6 CSVs loaded into memory\n")

# ── Column name cleanup ──────────────────────────────────────
# PostgreSQL does not accept spaces or special chars in col names
for df in [zones_df, drivers_df, weather_df,
           events_df, stations_df, rides_df]:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('(', '')
        .str.replace(')', '')
    )

# ── Drop computed columns not in schema ─────────────────────
if 'date_str' in weather_df.columns:
    weather_df.drop(columns=['date_str'], inplace=True)
if 'date_str' in events_df.columns:
    events_df.drop(columns=['date_str'], inplace=True)

# ── Loading config: table name, dataframe, load order ───────
# Order matters - zones must load before drivers (FK constraint)
# drivers must load before rides (FK constraint)
load_order = [
    ("zones",             zones_df,    200),
    ("drivers",           drivers_df,  2000),
    ("weather",           weather_df,  8784),
    ("events",            events_df,   132),
    ("charging_stations", stations_df, 85),
    ("rides",             rides_df,    500000),
]

print("=" * 60)
print("  LOADING DATA INTO PostgreSQL")
print("=" * 60)

total_start = time.time()
all_success = True

for table_name, df, expected_rows in load_order:
    start = time.time()

    try:
        # Use chunksize for large tables
        chunksize = 10000 if len(df) > 100000 else 1000

        df.to_sql(
            name      = table_name,
            con       = engine,
            if_exists = "append",   # schema already created
            index     = False,
            chunksize = chunksize,
            method    = "multi",    # faster batch inserts
        )

        elapsed = time.time() - start
        print(f"  ✓ {table_name:<22} "
              f"{len(df):>8,} rows  "
              f"({elapsed:.1f}s)")

    except Exception as e:
        print(f"  ✗ {table_name:<22} FAILED — {e}")
        all_success = False

total_elapsed = time.time() - total_start

print("=" * 60)
print(f"  Total Load Time : {total_elapsed:.1f} seconds")
print(f"  All Tables OK   : {all_success}")
print("=" * 60)
print("\n✓ Block 3 complete - all data loaded into PostgreSQL.")

Loading CSVs from disk...
  ✓ All 6 CSVs loaded into memory

  LOADING DATA INTO PostgreSQL
  ✓ zones                       200 rows  (0.1s)
  ✓ drivers                   2,000 rows  (1.2s)
  ✓ weather                   8,784 rows  (5.0s)
  ✓ events                      132 rows  (0.1s)
  ✓ charging_stations            85 rows  (0.1s)
  ✓ rides                   500,000 rows  (574.7s)
  Total Load Time : 581.2 seconds
  All Tables OK   : True

✓ Block 3 complete - all data loaded into PostgreSQL.


In [5]:
# ============================================================
# BLOCK 4 — PostgreSQL Data Verification
# ============================================================

verification_queries = {
    "zones":             "SELECT COUNT(*) FROM zones;",
    "drivers":           "SELECT COUNT(*) FROM drivers;",
    "weather":           "SELECT COUNT(*) FROM weather;",
    "events":            "SELECT COUNT(*) FROM events;",
    "charging_stations": "SELECT COUNT(*) FROM charging_stations;",
    "rides":             "SELECT COUNT(*) FROM rides;",
}

expected = {
    "zones":             200,
    "drivers":           2000,
    "weather":           8784,
    "events":            132,
    "charging_stations": 85,
    "rides":             500000,
}

print("=" * 60)
print("  PostgreSQL DATA VERIFICATION")
print("=" * 60)

all_ok = True

with engine.connect() as conn_eng:
    for table, query in verification_queries.items():
        result = conn_eng.execute(text(query)).fetchone()[0]
        exp    = expected[table]
        status = "✓" if result == exp else "⚠"
        if result != exp:
            all_ok = False
        print(f"  {status} {table:<22} "
              f"{result:>8,} rows  "
              f"(expected {exp:,})")

print("=" * 60)
print(f"  All Tables Verified : {all_ok}")
print("=" * 60)

# ── Quick business query test ────────────────────────────────
print("\n  QUICK QUERY TESTS")
print("  " + "-" * 50)

with engine.connect() as conn_eng:

    # Test 1 — Completion rate
    q1 = """
        SELECT
            status,
            COUNT(*)                          AS ride_count,
            ROUND(COUNT(*) * 100.0
                / SUM(COUNT(*)) OVER(), 2)    AS pct
        FROM rides
        GROUP BY status
        ORDER BY ride_count DESC;
    """
    result1 = pd.read_sql(text(q1), conn_eng)
    print("\n  Ride Status Distribution:")
    print(result1.to_string(index=False))

    # Test 2 — Top 5 zones by ride volume
    q2 = """
        SELECT
            pickup_zone_name,
            pickup_zone_type,
            COUNT(*) AS total_rides
        FROM rides
        GROUP BY pickup_zone_name, pickup_zone_type
        ORDER BY total_rides DESC
        LIMIT 5;
    """
    result2 = pd.read_sql(text(q2), conn_eng)
    print("\n  Top 5 Pickup Zones:")
    print(result2.to_string(index=False))

    # Test 3 — Revenue by vehicle type
    q3 = """
        SELECT
            vehicle_type,
            COUNT(*)                    AS rides,
            ROUND(AVG(fare_amount), 0)  AS avg_fare,
            ROUND(SUM(fare_amount), 0)  AS total_revenue
        FROM rides
        WHERE status = 'completed'
        GROUP BY vehicle_type
        ORDER BY total_revenue DESC;
    """
    result3 = pd.read_sql(text(q3), conn_eng)
    print("\n  Revenue by Vehicle Type:")
    print(result3.to_string(index=False))

    # Test 4 — EV charging utilization by network
    q4 = """
        SELECT
            network,
            COUNT(*)                         AS stations,
            ROUND(AVG(avg_utilization)*100,1) AS avg_util_pct,
            SUM(monthly_revenue_inr)          AS monthly_revenue
        FROM charging_stations
        WHERE is_operational = TRUE
        GROUP BY network
        ORDER BY monthly_revenue DESC;
    """
    result4 = pd.read_sql(text(q4), conn_eng)
    print("\n  EV Network Utilization:")
    print(result4.to_string(index=False))

print("\n" + "=" * 60)
print("  ✓ NOTEBOOK 02 COMPLETE - PostgreSQL fully loaded")
print("  Next  →  Notebook 03 - SQL Analytical Layer")
print("=" * 60)

  PostgreSQL DATA VERIFICATION
  ✓ zones                       200 rows  (expected 200)
  ✓ drivers                   2,000 rows  (expected 2,000)
  ✓ weather                   8,784 rows  (expected 8,784)
  ✓ events                      132 rows  (expected 132)
  ✓ charging_stations            85 rows  (expected 85)
  ✓ rides                   500,000 rows  (expected 500,000)
  All Tables Verified : True

  QUICK QUERY TESTS
  --------------------------------------------------

  Ride Status Distribution:
             status  ride_count   pct
          completed      389828 77.97
 cancelled_by_rider       49949  9.99
cancelled_by_driver       35091  7.02
    no_driver_found       25132  5.03

  Top 5 Pickup Zones:
    pickup_zone_name pickup_zone_type  total_rides
    HSR Layout Sec 7        tech_park         3721
   Manyata Tech Park        tech_park         3694
                ITPL        tech_park         3678
Embassy Tech Village        tech_park         3672
      Koramangala 1B